In [ ]:
#!/usr/bin/env python
# coding: utf-8

import numpy as np
from scipy.optimize import least_squares
from scipy.integrate import quad

from Options_Math_Helpers import *
from Options_Math_Black_Scholes import *

ombs = OptionsMathBlackScholes()


class HestonSkew(OptionsMathHelpers):

    def __init__(self):
        pass

    # -------------------------
    # Heston characteristic fn
    # -------------------------

    def _char_func(self, u, fwd, time, v0, kappa, theta, sigma, rho):
        i = 1j

        d = np.sqrt((rho * sigma * i * u - kappa)**2 +
                    sigma * sigma * (i * u + u * u))

        g = (kappa - rho * sigma * i * u - d) / \
            (kappa - rho * sigma * i * u + d)

        exp_dt = np.exp(-d * time)

        C = (kappa * theta / (sigma * sigma)) * (
            (kappa - rho * sigma * i * u - d) * time -
            2.0 * np.log((1 - g * exp_dt) / (1 - g))
        )

        D = ((kappa - rho * sigma * i * u - d) / (sigma * sigma)) * \
            ((1 - exp_dt) / (1 - g * exp_dt))

        return np.exp(C + D * v0 + i * u * np.log(fwd))

    # -------------------------
    # Option price
    # -------------------------

    def _heston_prob(self, phi, fwd, strike, time, params, j):
        v0, kappa, theta, sigma, rho = params

        i = 1j
        u = phi - i if j == 1 else phi

        cf = self._char_func(u, fwd, time, v0, kappa, theta, sigma, rho)

        numerator = np.exp(-i * phi * np.log(strike)) * cf
        denominator = i * phi

        return np.real(numerator / denominator)

    def heston_call_price(self, fwd, strike, time, params, limit=100.0):
        P1 = 0.5 + (1 / np.pi) * quad(
            lambda phi: self._heston_prob(phi, fwd, strike, time, params, 1),
            0, limit
        )[0]

        P2 = 0.5 + (1 / np.pi) * quad(
            lambda phi: self._heston_prob(phi, fwd, strike, time, params, 2),
            0, limit
        )[0]

        return fwd * P1 - strike * P2

    # -------------------------
    # Implied volatility
    # -------------------------

    def heston_vol(self, fwd=None, strike=None, time=None,
                   v0=None, kappa=None, theta=None, sigma=None, rho=None,
                   **kwargs):

        fwd, strike, time = self.to_arrays(fwd, strike, time)

        vols = np.zeros_like(strike, dtype=float)

        params = (v0, kappa, theta, sigma, rho)

        for i, K in enumerate(strike):
            price = self.heston_call_price(fwd[i], K, time[i], params)
            vols[i] = ombs.implied_vol(
                opt_type='call',
                fwd_value=fwd[i],
                strike=K,
                time_to_expiry=time[i],
                price=price
            )

        return vols

    # -------------------------
    # Calibration
    # -------------------------

    def _heston_vol_weighted(self, params, fwd, strike, time, target_vols, weights):
        v0, kappa, theta, sigma, rho = params

        model_vols = self.heston_vol(fwd=fwd,
                                     strike=strike,
                                     time=time,
                                     v0=v0,
                                     kappa=kappa,
                                     theta=theta,
                                     sigma=sigma,
                                     rho=rho)

        return weights * (model_vols - target_vols)

    def calibrate_heston_weighted(self,
                                  fwd=None, strike=None, time=None,
                                  target_vols=None,
                                  weighting='vega',
                                  weights=None,
                                  weight_eps=1e-8,
                                  initial_guess=(0.04, 1.0, 0.04, 0.5, -0.5),
                                  bounds=([1e-6, 1e-3, 1e-6, 1e-3, -0.999],
                                          [2.0, 10.0, 2.0, 5.0, 0.999]),
                                  **lsq_kwargs):

        fwd, strike, time, target_vols, weights, weight_eps, initial_guess = \
            self.to_arrays(fwd, strike, time, target_vols,
                           weights, weight_eps, initial_guess)

        target_vols = np.nan_to_num(target_vols, nan=weight_eps)

        if weighting in ('vega', 'sqrt', 'norm'):
            weights = np.nan_to_num(weights, nan=weight_eps)
            weights = np.maximum(weights, weight_eps)

            if weighting == 'sqrt':
                weights = np.sqrt(weights)
            elif weighting == 'norm':
                weights = weights / np.max(weights)
        else:
            raise ValueError("Unknown weighting")

        obj_fn = lambda params: self._heston_vol_weighted(
            params, fwd, strike, time, target_vols, weights
        )

        result = least_squares(obj_fn,
                               initial_guess,
                               bounds=bounds,
                               **lsq_kwargs)

        v0, kappa, theta, sigma, rho = result.x

        return v0, kappa, theta, sigma, rho, result
